# Subcontractor Counts By Month

This notebook reads the wide billing output and builds a month-by-month count table using `Date_Delivered_to_VA`.

- Rows: `Subcontractor`
- Columns: month derived from `Date_Delivered_to_VA`


In [ ]:
import csv
from collections import defaultdict
from datetime import datetime
from pathlib import Path


In [ ]:
def resolve_input_csv_path() -> Path:
    candidates = [
        Path("output/HITL_billing_wide.csv"),
        Path("tmp_notebook_check/output/HITL_billing_wide.csv"),
    ]
    for candidate in candidates:
        if candidate.exists():
            return candidate
    return candidates[0]


INPUT_CSV_PATH = resolve_input_csv_path()
INPUT_CSV_PATH


In [ ]:
counts_by_subcontractor = defaultdict(lambda: defaultdict(int))
months = set()

with INPUT_CSV_PATH.open("r", encoding="utf-8-sig", newline="") as handle:
    reader = csv.DictReader(handle)
    for row in reader:
        delivered_date = (row.get("Date_Delivered_to_VA") or "").strip()
        if not delivered_date:
            continue

        subcontractor = (row.get("Subcontractor") or "").strip() or "N/A"
        month = datetime.strptime(delivered_date, "%Y-%m-%d").strftime("%Y-%m")

        counts_by_subcontractor[subcontractor][month] += 1
        months.add(month)

sorted_months = sorted(months)
preferred_subcontractor_order = ["XBP", "IRM", "TSS", "VA", "N/A"]
sorted_subcontractors = [
    subcontractor
    for subcontractor in preferred_subcontractor_order
    if subcontractor in counts_by_subcontractor
] + sorted(
    subcontractor
    for subcontractor in counts_by_subcontractor
    if subcontractor not in preferred_subcontractor_order
)

counts_table = []
for subcontractor in sorted_subcontractors:
    result_row = {"Subcontractor": subcontractor}
    for month in sorted_months:
        result_row[month] = counts_by_subcontractor[subcontractor].get(month, 0)
    counts_table.append(result_row)

print(f"Input CSV: {INPUT_CSV_PATH.resolve()}")
print(f"Months found: {sorted_months}")
print(f"Subcontractors found: {sorted_subcontractors}")


In [ ]:
counts_table
